In [1]:
import os
import cv2 as cv
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from pathlib import Path
import sys

In [ ]:
class BrailleCNN(nn.Module):
    def __init__(self, num_classes=26):
        super(BrailleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x))) 
        x = self.pool(torch.relu(self.conv2(x)))  
        x = x.view(-1, 64 * 7 * 7)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

In [3]:
def preprocess_image(img: np.ndarray) -> np.ndarray:
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    clahe = cv.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    equalized = clahe.apply(gray)
    denoised = cv.bilateralFilter(equalized, 9, 100, 100)
    thresh = cv.adaptiveThreshold(denoised, 255,
                                   cv.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv.THRESH_BINARY_INV,
                                   blockSize=5, C=2)
    return thresh

In [4]:
class YOLOBrailleDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.transform = transform
        self.count = 0
        self.image_paths = list(self.image_dir.glob("*.jpg")) + list(self.image_dir.glob("*.png"))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = cv.imread(str(image_path))
        if image is None:
            raise ValueError(f"Cannot read image: {image_path}")
        h, w = image.shape[:2]

        label_path = self.label_dir / (image_path.stem + ".txt")
        if not label_path.exists():
            return None

        Xs, Ys = [], []
        with open(label_path) as f:
            for line in f:
                values = line.strip().split()
                if len(values) != 5:
                    continue
                class_id, x, y, bw, bh = map(float, values)
                xmin = int((x - bw / 2) * w)
                ymin = int((y - bh / 2) * h)
                xmax = int((x + bw / 2) * w)
                ymax = int((y + bh / 2) * h)

                if xmin < 0 or ymin < 0 or xmax > w or ymax > h or xmin >= xmax or ymin >= ymax:
                    continue

                cropped = image[ymin:ymax, xmin:xmax]
                if cropped.size == 0:
                    continue

                processed = preprocess_image(cropped)
                if self.transform:
                    try:
                        processed = self.transform(processed)
                    except Exception as e:
                        continue

                if processed.shape != (1, 28, 28):
                    continue

                Xs.append(processed)
                Ys.append(int(class_id))

        if len(Xs) == 0 or len(Ys) == 0:
            self.count += 1
            return None

        return torch.stack(Xs), torch.tensor(Ys)

In [5]:
def collate_fn(batch):
    X, Y = [], []
    for b in batch:
        if b is None or len(b) != 2:
            continue
        xb, yb = b
        if xb is None or yb is None or len(xb) == 0:
            continue
        X.extend(xb)
        Y.extend(yb)

    if len(X) == 0:
        return torch.empty(0), torch.empty(0, dtype=torch.long)
    return torch.stack(X), torch.tensor(Y)

In [6]:
train_img_dir = r"C:\Users\veerk\Downloads\braille.v2i.yolov11\train\images"
train_lbl_dir = r"C:\Users\veerk\Downloads\braille.v2i.yolov11\train\labels"

valid_img_dir = r"C:\Users\veerk\Downloads\braille.v2i.yolov11\valid\images"
valid_lbl_dir = r"C:\Users\veerk\Downloads\braille.v2i.yolov11\valid\labels"

test_img_dir = r"C:\Users\veerk\Downloads\braille.v2i.yolov11\test\images"
test_lbl_dir = r"C:\Users\veerk\Downloads\braille.v2i.yolov11\test\labels"

In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((28, 28)),
    transforms.ToTensor()
])

train_dataset = YOLOBrailleDataset(train_img_dir, train_lbl_dir, transform)
valid_dataset = YOLOBrailleDataset(valid_img_dir, valid_lbl_dir, transform)
test_dataset  = YOLOBrailleDataset(test_img_dir, test_lbl_dir, transform)

In [8]:
if len(train_dataset) == 0:
    print(f"Error: No images (.jpg or .png) found in the training directory: '{train_img_dir}'")
    print("Please ensure the path is correct and the directory contains image files.")
    sys.exit(1)
if len(valid_dataset) == 0:
    print(f"Warning: No images found in the validation directory: '{valid_img_dir}'")

if len(test_dataset) == 0:
    print(f"Warning: No images found in the test directory: '{test_img_dir}'")

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, collate_fn=collate_fn)


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BrailleCNN(num_classes=64).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [10]:
def train(model, loader, optimizer, criterion):
    model.train()
    total, correct, loss_val = 0, 0, 0
    for X, Y in loader:
        if X.shape[0] == 0: continue
        X, Y = X.to(device), Y.to(device)
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, Y)
        loss.backward()
        optimizer.step()

        loss_val += loss.item()
        _, predicted = outputs.max(1)
        total += Y.size(0)
        correct += predicted.eq(Y).sum().item()

    if total == 0: return 0, 0
    return loss_val / len(loader), 100 * correct / total

In [11]:
def evaluate(model, loader):
    model.eval()
    total, correct = 0, 0
    with torch.no_grad():
        for X, Y in loader:
            if X.shape[0] == 0: continue
            X, Y = X.to(device), Y.to(device)
            outputs = model(X)
            _, predicted = outputs.max(1)
            total += Y.size(0)
            correct += predicted.eq(Y).sum().item()
    if total == 0: return 0
    return 100 * correct / total

In [12]:
epochs = 16
for epoch in range(epochs):
    loss, acc = train(model, train_loader, optimizer, criterion)
    val_acc = evaluate(model, valid_loader)
    print(f"Epoch {epoch+1:02d}: Train Loss={loss:.4f}, Train Acc={acc:.2f}%, Valid Acc={val_acc:.2f}%")

test_acc = evaluate(model, test_loader)
print(f"\nTest Accuracy: {test_acc:.2f}%")

Epoch 01: Train Loss=2.1263, Train Acc=42.26%, Valid Acc=63.22%
Epoch 02: Train Loss=1.1416, Train Acc=67.53%, Valid Acc=67.66%
Epoch 03: Train Loss=1.0033, Train Acc=71.26%, Valid Acc=72.76%
Epoch 04: Train Loss=0.9124, Train Acc=74.29%, Valid Acc=74.43%
Epoch 05: Train Loss=0.8566, Train Acc=75.77%, Valid Acc=75.66%
Epoch 06: Train Loss=0.8095, Train Acc=77.07%, Valid Acc=76.90%
Epoch 07: Train Loss=0.7742, Train Acc=78.11%, Valid Acc=77.34%
Epoch 08: Train Loss=0.7394, Train Acc=79.08%, Valid Acc=78.32%
Epoch 09: Train Loss=0.7140, Train Acc=79.74%, Valid Acc=78.95%
Epoch 10: Train Loss=0.6825, Train Acc=80.64%, Valid Acc=79.55%
Epoch 11: Train Loss=0.6645, Train Acc=81.24%, Valid Acc=79.62%
Epoch 12: Train Loss=0.6427, Train Acc=81.74%, Valid Acc=80.13%
Epoch 13: Train Loss=0.6290, Train Acc=82.03%, Valid Acc=80.55%
Epoch 14: Train Loss=0.6120, Train Acc=82.51%, Valid Acc=80.09%
Epoch 15: Train Loss=0.5936, Train Acc=82.82%, Valid Acc=80.89%
Epoch 16: Train Loss=0.5842, Train Acc=8